# Section 3 · CSV와 Excel 파일 읽고 쓰기

데이터를 읽는 것부터 자료형·결측값·호가 조건을 검사하고, 계산 결과를 저장한 뒤 다시 검증하는 과정까지 실습합니다.

**준비:** 이 노트북과 같은 폴더 안에 `data` 폴더가 있어야 합니다. 압축을 풀 때 폴더 구조를 유지하세요. 앞 장을 실행하지 않은 새 커널에서 위에서부터 모두 실행할 수 있습니다. 새 파일은 `output` 폴더에 저장합니다.

사용하는 옵션 호가는 강의록에 포함된 **교육용 합성 자료**이며, CSV와 Excel에 동일한 42개 계약이 들어 있습니다. 금리·변동성은 백분율이 아닌 소수 단위, 잔존만기는 연 단위입니다.

| 파일 | 용도 |
| --- | --- |
| `data/european_option_market_data.csv` | 옵션 호가 원본 |
| `data/european_option_market_data.xlsx` | 같은 호가와 변수 설명·안내 시트 |
| `output/*` | 이 노트북을 실행하면서 만드는 결과 파일 |

강의록의 범용 `df`, `input_path`, `output_path` 예제는 실제 실습 자료와 연결해 실행합니다. Stata 예제에는 별도로 표시한 작은 합성 자료를 만들어 사용합니다.

## 실습 폴더 구성하기

강의록의 예시 폴더 구조입니다. 배포본에서는 노트북 이름만 `section03_data_io.ipynb`로 되어 있으며 `data`와 `output`의 상대적 위치는 같습니다.

*강의록: `section03_data_io.tex` 148행 · 03-L001*

```text
python_economics/
|-- 03_reading_writing_data.ipynb
|-- data/
|   |-- european_option_market_data.csv
|   `-- european_option_market_data.xlsx
`-- output/
```

## pandas와 Excel 지원 패키지 준비하기

패키지가 없을 때만 별도 코드 셀에 입력하는 설치 명령입니다. 현재 커널과 같은 환경에 설치되도록 Notebook의 `%pip`를 사용합니다.

*강의록: `section03_data_io.tex` 179행 · 03-L002*

```python
%pip install pandas openpyxl
```

**이번 장에서 사용할 패키지 불러오기**



*강의록: `section03_data_io.tex` 186행 · 03-L003*

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

print("pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

pandas version: 2.2.3
NumPy version: 2.3.5


## 파일 경로를 먼저 확인하기

**데이터 파일의 상대경로 지정하기**

`False`가 나오면 다음 셀로 넘어가기 전에 압축을 푼 폴더와 데이터 파일의 위치를 확인합니다.

*강의록: `section03_data_io.tex` 212행 · 03-L004*

In [2]:
from pathlib import Path

notebook_dir = Path.cwd()
data_dir = Path("data")
output_dir = Path("output")

csv_path = data_dir / "european_option_market_data.csv"
excel_path = data_dir / "european_option_market_data.xlsx"

print("현재 작업 폴더:", notebook_dir)
print("CSV 경로:", csv_path)
print("CSV 존재 여부:", csv_path.exists())
print("Excel 존재 여부:", excel_path.exists())

현재 작업 폴더: /workspace/scratch/9f1663ff71ca/deliverable/python_economics_notebooks
CSV 경로: data/european_option_market_data.csv
CSV 존재 여부: True
Excel 존재 여부: True


<details><summary>강의록의 참고 출력 · 03-L005</summary>

```text
CSV 존재 여부: True
Excel 존재 여부: True
```

</details>

결과를 저장할 폴더를 준비합니다. 폴더가 이미 있어도 오류가 나지 않습니다.

*강의록: `section03_data_io.tex` 239행 · 03-L006*

In [3]:
output_dir.mkdir(parents=True, exist_ok=True)

## CSV 파일의 구조 이해하기

CSV 원본의 앞부분을 축약한 모양입니다. 쉼표는 열을 구분하며 첫 줄은 열 이름입니다.

*강의록: `section03_data_io.tex` 261행 · 03-L007*

```text
contract_id,quote_date,underlying,currency,option_type,...
SYNTH100_20261208_C_080,2026-09-08,SYNTH100,KRW,call,...
SYNTH100_20261208_P_080,2026-09-08,SYNTH100,KRW,put,...
```

## CSV 파일을 가장 간단하게 읽기

마지막 줄에 DataFrame을 놓으면 Notebook에서 표 형태로 표시됩니다.

*강의록: `section03_data_io.tex` 294행 · 03-L008*

In [4]:
quotes_auto = pd.read_csv(csv_path, encoding="utf-8-sig")

quotes_auto.head()

,contract_id,quote_date,underlying,currency,option_type,maturity_date,time_to_maturity,spot_price,strike_price,risk_free_rate,dividend_yield,implied_volatility,bid_price,ask_price,mid_price,trading_volume,open_interest,data_source
0,SYNTH100_20261208_C_080,2026-09-08,SYNTH100,KRW,call,2026-12-08,0.249315,100.0,80.0,0.035,0.0125,0.239763,20.409310,20.586972,20.498141,724,3924,synthetic_black_scholes
1,SYNTH100_20261208_P_080,2026-09-08,SYNTH100,KRW,put,2026-12-08,0.249315,100.0,80.0,0.035,0.0125,0.239763,0.101863,0.126646,0.114254,706,3804,synthetic_black_scholes
2,SYNTH100_20261208_C_090,2026-09-08,SYNTH100,KRW,call,2026-12-08,0.249315,100.0,90.0,0.035,0.0125,0.218311,11.245520,11.349466,11.297493,834,4054,synthetic_black_scholes
3,SYNTH100_20261208_P_090,2026-09-08,SYNTH100,KRW,put,2026-12-08,0.249315,100.0,90.0,0.035,0.0125,0.218311,0.814018,0.839433,0.826726,816,3934,synthetic_black_scholes
4,SYNTH100_20261208_C_095,2026-09-08,SYNTH100,KRW,call,2026-12-08,0.249315,100.0,95.0,0.035,0.0125,0.210787,7.368690,7.441279,7.404985,889,4119,synthetic_black_scholes


**DataFrame의 기본 구조 확인하기**



*강의록: `section03_data_io.tex` 307행 · 03-L009*

In [5]:
print("shape:", quotes_auto.shape)
print("columns:", quotes_auto.columns.tolist())
print("\nfirst three rows:")
print(quotes_auto.head(3))
print("\nlast three rows:")
print(quotes_auto.tail(3))

shape: (42, 18)
columns: ['contract_id', 'quote_date', 'underlying', 'currency', 'option_type', 'maturity_date', 'time_to_maturity', 'spot_price', 'strike_price', 'risk_free_rate', 'dividend_yield', 'implied_volatility', 'bid_price', 'ask_price', 'mid_price', 'trading_volume', 'open_interest', 'data_source']

first three rows:
               contract_id  quote_date underlying currency option_type  \
0  SYNTH100_20261208_C_080  2026-09-08   SYNTH100      KRW        call   
1  SYNTH100_20261208_P_080  2026-09-08   SYNTH100      KRW         put   
2  SYNTH100_20261208_C_090  2026-09-08   SYNTH100      KRW        call   

  maturity_date  time_to_maturity  spot_price  strike_price  risk_free_rate  \
0    2026-12-08          0.249315       100.0          80.0           0.035   
1    2026-12-08          0.249315       100.0          80.0           0.035   
2    2026-12-08          0.249315       100.0          90.0           0.035   

   dividend_yield  implied_volatility  bid_price  ask_pri

<details><summary>강의록의 참고 출력 · 03-L010</summary>

```text
shape: (42, 18)
```

</details>

## 자료형과 날짜를 명시하여 CSV 읽기

**열별 자료형을 지정하여 CSV 읽기**

연속형 수치와 문자열의 자료형을 명시하고 날짜 열은 따로 해석합니다.

*강의록: `section03_data_io.tex` 347행 · 03-L011*

In [6]:
dtype_map = {
    "contract_id": "string",
    "underlying": "string",
    "currency": "string",
    "option_type": "string",
    "time_to_maturity": float,
    "spot_price": float,
    "strike_price": float,
    "risk_free_rate": float,
    "dividend_yield": float,
    "implied_volatility": float,
    "bid_price": float,
    "ask_price": float,
    "mid_price": float,
    "trading_volume": int,
    "open_interest": int,
    "data_source": "string",
}

date_columns = ["quote_date", "maturity_date"]

quotes = pd.read_csv(
    csv_path,
    encoding="utf-8-sig",
    dtype=dtype_map,
    parse_dates=date_columns,
)

**읽은 뒤 실제 자료형 확인하기**



*강의록: `section03_data_io.tex` 380행 · 03-L012*

In [7]:
print(quotes.dtypes)

contract_id           string[python]
quote_date            datetime64[ns]
underlying            string[python]
currency              string[python]
option_type           string[python]
maturity_date         datetime64[ns]
time_to_maturity             float64
spot_price                   float64
strike_price                 float64
risk_free_rate               float64
dividend_yield               float64
implied_volatility           float64
bid_price                    float64
ask_price                    float64
mid_price                    float64
trading_volume                 int64
open_interest                  int64
data_source           string[python]
dtype: object


가격·금리·변동성 등의 열이 모두 실수형인지 검사합니다. 성공하면 오류 없이 다음 셀로 넘어갑니다.

*강의록: `section03_data_io.tex` 386행 · 03-L013*

In [8]:
float_columns = [
    "time_to_maturity",
    "spot_price",
    "strike_price",
    "risk_free_rate",
    "dividend_yield",
    "implied_volatility",
    "bid_price",
    "ask_price",
    "mid_price",
]

print(quotes[float_columns].dtypes)

assert (quotes[float_columns].dtypes == np.dtype(float)).all()

time_to_maturity      float64
spot_price            float64
strike_price          float64
risk_free_rate        float64
dividend_yield        float64
implied_volatility    float64
bid_price             float64
ask_price             float64
mid_price             float64
dtype: object


### 숫자처럼 보이는 문자열 정리하기

**퍼센트 문자열을 실수 비율로 변환하기**

퍼센트 기호를 제거한 후 100으로 나누어 소수 단위로 바꿉니다.

*강의록: `section03_data_io.tex` 424행 · 03-L014*

In [9]:
rate_text = pd.Series(["3.5%", "3.6%", "3.75%"], dtype="string")
rates = rate_text.str.rstrip("%").astype(float) / 100.0

print(rates)
print(rates.dtype)

0    0.0350
1    0.0360
2    0.0375
dtype: float64
float64


<details><summary>강의록의 참고 출력 · 03-L015</summary>

```text
0    0.0350
1    0.0360
2    0.0375
dtype: float64
```

</details>

## 읽은 직후 반드시 확인할 내용

### 구조와 요약정보 확인

*강의록: `section03_data_io.tex` 457행 · 03-L016*

In [10]:
print("number of rows:", quotes.shape[0])
print("number of columns:", quotes.shape[1])
print("\ncolumn names:")
print(quotes.columns.tolist())

quotes.info()

number of rows: 42
number of columns: 18

column names:
['contract_id', 'quote_date', 'underlying', 'currency', 'option_type', 'maturity_date', 'time_to_maturity', 'spot_price', 'strike_price', 'risk_free_rate', 'dividend_yield', 'implied_volatility', 'bid_price', 'ask_price', 'mid_price', 'trading_volume', 'open_interest', 'data_source']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   contract_id         42 non-null     string        
 1   quote_date          42 non-null     datetime64[ns]
 2   underlying          42 non-null     string        
 3   currency            42 non-null     string        
 4   option_type         42 non-null     string        
 5   maturity_date       42 non-null     datetime64[ns]
 6   time_to_maturity    42 non-null     float64       
 7   spot_price          42 non-null     float64    

*강의록: `section03_data_io.tex` 468행 · 03-L017*

In [11]:
summary = quotes[
    [
        "spot_price",
        "strike_price",
        "risk_free_rate",
        "implied_volatility",
        "mid_price",
    ]
].describe()

summary

,spot_price,strike_price,risk_free_rate,implied_volatility,mid_price
count,42.0,42.000000,42.000000,42.000000,42.000000
mean,100.0,100.000000,0.036167,0.219628,8.095425
std,0.0,12.395908,0.001040,0.016698,6.607171
min,100.0,80.000000,0.035000,0.195078,0.114254
25%,100.0,90.000000,0.035000,0.207740,2.779546
50%,100.0,100.000000,0.036000,0.217740,6.599450
75%,100.0,110.000000,0.037500,0.228311,11.332275
max,100.0,120.000000,0.037500,0.259763,23.745749


### 결측값과 중복값 확인

**결측값과 중복 계약 확인하기**

누락된 값과 중복 계약이 없더라도 다음 셀에서 금융자료의 조건을 따로 검사합니다.

*강의록: `section03_data_io.tex` 484행 · 03-L018*

In [12]:
missing_by_column = quotes.isna().sum()
print(missing_by_column)

number_of_missing_values = int(missing_by_column.sum())
number_of_duplicate_contracts = int(quotes.duplicated(subset="contract_id").sum())

print("total missing values:", number_of_missing_values)
print("duplicate contract IDs:", number_of_duplicate_contracts)

contract_id           0
quote_date            0
underlying            0
currency              0
option_type           0
maturity_date         0
time_to_maturity      0
spot_price            0
strike_price          0
risk_free_rate        0
dividend_yield        0
implied_volatility    0
bid_price             0
ask_price             0
mid_price             0
trading_volume        0
open_interest         0
data_source           0
dtype: int64
total missing values: 0
duplicate contract IDs: 0


<details><summary>강의록의 참고 출력 · 03-L019</summary>

```text
total missing values: 0
duplicate contract IDs: 0
```

</details>

### 금융자료의 경제적 제약조건 확인

**옵션 호가의 기본 제약조건 검사하기**

`bid ≤ mid ≤ ask`와 양수 입력 조건을 확인합니다.

*강의록: `section03_data_io.tex` 509행 · 03-L020*

In [13]:
valid_quote_order = (
    (quotes["bid_price"] <= quotes["mid_price"])
    & (quotes["mid_price"] <= quotes["ask_price"])
)

positive_inputs = (
    (quotes["time_to_maturity"] > 0.0) & (quotes["spot_price"] > 0.0)
    & (quotes["strike_price"] > 0.0)
    & (quotes["implied_volatility"] > 0.0)
)

if not valid_quote_order.all():
    raise ValueError("bid <= mid <= ask 조건을 위반한 행이 있습니다.")

if not positive_inputs.all():
    raise ValueError("양수여야 하는 입력값에 문제가 있습니다.")

print("Basic validation passed.")

Basic validation passed.


<details><summary>강의록의 참고 출력 · 03-L021</summary>

```text
Basic validation passed.
```

</details>

## 필요한 행과 열을 선택하기

**콜옵션 선택과 새 변수 계산**

필요한 행·열을 고른 뒤 `.copy()`로 독립된 표를 만들고 호가 스프레드와 moneyness를 추가합니다.

*강의록: `section03_data_io.tex` 550행 · 03-L022*

In [14]:
call_quotes = quotes.loc[
    quotes["option_type"] == "call",
    [
        "contract_id",
        "quote_date",
        "maturity_date",
        "time_to_maturity",
        "spot_price",
        "strike_price",
        "implied_volatility",
        "bid_price",
        "ask_price",
        "mid_price",
    ],
].copy()

call_quotes["bid_ask_spread"] = call_quotes["ask_price"] - call_quotes["bid_price"]

call_quotes["moneyness"] = call_quotes["strike_price"] / call_quotes["spot_price"]

print(call_quotes.shape)
call_quotes.head()

(21, 12)


,contract_id,quote_date,maturity_date,time_to_maturity,spot_price,strike_price,implied_volatility,bid_price,ask_price,mid_price,bid_ask_spread,moneyness
0,SYNTH100_20261208_C_080,2026-09-08,2026-12-08,0.249315,100.0,80.0,0.239763,20.409310,20.586972,20.498141,0.177662,0.80
2,SYNTH100_20261208_C_090,2026-09-08,2026-12-08,0.249315,100.0,90.0,0.218311,11.245520,11.349466,11.297493,0.103946,0.90
4,SYNTH100_20261208_C_095,2026-09-08,2026-12-08,0.249315,100.0,95.0,0.210787,7.368690,7.441279,7.404985,0.072589,0.95
6,SYNTH100_20261208_C_100,2026-09-08,2026-12-08,0.249315,100.0,100.0,0.205000,4.318815,4.366384,4.342600,0.047569,1.00
8,SYNTH100_20261208_C_105,2026-09-08,2026-12-08,0.249315,100.0,105.0,0.200716,2.227498,2.261283,2.244391,0.033785,1.05


<details><summary>강의록의 참고 출력 · 03-L023</summary>

```text
(21, 12)
```

</details>

## CSV 파일로 저장하기

원본과 다른 경로에 CSV를 저장합니다.

*강의록: `section03_data_io.tex` 596행 · 03-L024*

In [15]:
output_csv_path = ( output_dir / "european_call_quotes_with_features.csv" )
call_quotes.to_csv( output_csv_path, index=False, encoding="utf-8-sig", )

print(output_csv_path)
print(output_csv_path.exists())

output/european_call_quotes_with_features.csv
True


### 왜 index=False가 필요한가?

**index=False를 생략했을 때의 문제**

인덱스를 저장하면 다시 읽었을 때 불필요한 열이 생깁니다.

*강의록: `section03_data_io.tex` 619행 · 03-L025*

In [16]:
bad_index_path = output_dir / "bad_index_example.csv"

call_quotes.head(3).to_csv(bad_index_path, encoding="utf-8-sig")

bad_index_data = pd.read_csv(bad_index_path, encoding="utf-8-sig")

print(bad_index_data.columns[0])

Unnamed: 0


<details><summary>강의록의 참고 출력 · 03-L026</summary>

```text
Unnamed: 0
```

</details>

강의록의 범용 저장 구문입니다. 바로 앞의 `call_quotes`와 별도 출력 경로를 연결하여 그대로 실행합니다.

*강의록: `section03_data_io.tex` 638행 · 03-L027*

In [17]:
df = call_quotes
output_path = output_dir / "call_quotes_template_27.csv"

In [18]:
df.to_csv(output_path, index=False, encoding="utf-8-sig")

In [19]:
print("저장한 파일:", output_path)
print("존재 여부:", output_path.exists())

저장한 파일: output/call_quotes_template_27.csv
존재 여부: True


### 소수 자릿수를 함부로 줄이지 않기

**낮은 출력 정밀도로 금리가 훼손되는 예**

출력 정밀도를 낮추면 원래 금리 `0.035125`가 `0.04`로 바뀝니다. 화면 표시용 반올림과 계산자료 저장을 구분합니다.

*강의록: `section03_data_io.tex` 650행 · 03-L028*

In [20]:
rate_example = pd.DataFrame({"risk_free_rate": [0.035125]})
bad_rate_path = output_dir / "bad_rate_precision.csv"

rate_example.to_csv(bad_rate_path, index=False, float_format="%.2f")

reloaded_bad_rate = pd.read_csv(bad_rate_path)
print(reloaded_bad_rate.loc[0, "risk_free_rate"])

0.04


<details><summary>강의록의 참고 출력 · 03-L029</summary>

```text
0.04
```

</details>

## 저장한 CSV를 다시 읽어 확인하기

**저장한 결과 파일 재검증하기**

저장한 자료를 다시 읽을 때에도 자료형과 날짜 열을 지정합니다.

*강의록: `section03_data_io.tex` 683행 · 03-L030*

In [21]:
output_dtype_map = {
    "contract_id": "string",
    "time_to_maturity": float,
    "spot_price": float,
    "strike_price": float,
    "implied_volatility": float,
    "bid_price": float,
    "ask_price": float,
    "mid_price": float,
    "bid_ask_spread": float,
    "moneyness": float,
}

reloaded_calls = pd.read_csv(
    output_csv_path,
    encoding="utf-8-sig",
    dtype=output_dtype_map,
    parse_dates=["quote_date", "maturity_date"],
)

print(reloaded_calls.shape)
print(reloaded_calls.dtypes)

(21, 12)
contract_id           string[python]
quote_date            datetime64[ns]
maturity_date         datetime64[ns]
time_to_maturity             float64
spot_price                   float64
strike_price                 float64
implied_volatility           float64
bid_price                    float64
ask_price                    float64
mid_price                    float64
bid_ask_spread               float64
moneyness                    float64
dtype: object


파일 저장 전후의 표가 허용오차 안에서 같은지 실제로 검사합니다.

*강의록: `section03_data_io.tex` 710행 · 03-L031*

In [22]:
pd.testing.assert_frame_equal(
    call_quotes.reset_index(drop=True),
    reloaded_calls.reset_index(drop=True),
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("CSV round-trip check passed.")

CSV round-trip check passed.


<details><summary>강의록의 참고 출력 · 03-L032</summary>

```text
CSV round-trip check passed.
```

</details>

## 문자 인코딩과 한글 깨짐

UTF-8 BOM을 포함하는 `utf-8-sig`로 저장하는 범용 예제를 현재 실습 표에 적용합니다.

*강의록: `section03_data_io.tex` 740행 · 03-L033*

In [23]:
df = call_quotes
output_path = output_dir / "call_quotes_template_33.csv"

In [24]:
df.to_csv(output_path, index=False, encoding="utf-8-sig")

In [25]:
print("저장한 파일:", output_path)
print("존재 여부:", output_path.exists())

저장한 파일: output/call_quotes_template_33.csv
존재 여부: True


읽기 예제의 `input_path`를 방금 저장한 파일에 연결합니다.

*강의록: `section03_data_io.tex` 748행 · 03-L034*

In [26]:
input_path = output_path

In [27]:
df = pd.read_csv(input_path, encoding="utf-8-sig")

In [28]:
df.head(3)

,contract_id,quote_date,maturity_date,time_to_maturity,spot_price,strike_price,implied_volatility,bid_price,ask_price,mid_price,bid_ask_spread,moneyness
0,SYNTH100_20261208_C_080,2026-09-08,2026-12-08,0.249315,100.0,80.0,0.239763,20.40931,20.586972,20.498141,0.177662,0.80
1,SYNTH100_20261208_C_090,2026-09-08,2026-12-08,0.249315,100.0,90.0,0.218311,11.24552,11.349466,11.297493,0.103946,0.90
2,SYNTH100_20261208_C_095,2026-09-08,2026-12-08,0.249315,100.0,95.0,0.210787,7.36869,7.441279,7.404985,0.072589,0.95


**UTF-8과 CP949를 순서대로 시도하는 예**

UTF-8 읽기가 실패할 때에만 CP949를 시도합니다. 이번 실습 파일은 UTF-8이므로 첫 번째 읽기가 성공합니다.

*강의록: `section03_data_io.tex` 756행 · 03-L035*

In [29]:
try:
    df = pd.read_csv(input_path, encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(input_path, encoding="cp949")

In [30]:
df.head(3)

,contract_id,quote_date,maturity_date,time_to_maturity,spot_price,strike_price,implied_volatility,bid_price,ask_price,mid_price,bid_ask_spread,moneyness
0,SYNTH100_20261208_C_080,2026-09-08,2026-12-08,0.249315,100.0,80.0,0.239763,20.40931,20.586972,20.498141,0.177662,0.80
1,SYNTH100_20261208_C_090,2026-09-08,2026-12-08,0.249315,100.0,90.0,0.218311,11.24552,11.349466,11.297493,0.103946,0.90
2,SYNTH100_20261208_C_095,2026-09-08,2026-12-08,0.249315,100.0,95.0,0.210787,7.36869,7.441279,7.404985,0.072589,0.95


## Excel 파일 읽기

Excel 파일의 시트 이름을 먼저 확인합니다.

*강의록: `section03_data_io.tex` 779행 · 03-L036*

In [31]:
excel_file = pd.ExcelFile(excel_path, engine="openpyxl")

print(excel_file.sheet_names)

['option_quotes', 'data_dictionary', 'README']


<details><summary>강의록의 참고 출력 · 03-L037</summary>

```text
['option_quotes', 'data_dictionary', 'README']
```

</details>

**Excel의 특정 시트를 DataFrame으로 읽기**



*강의록: `section03_data_io.tex` 794행 · 03-L038*

In [32]:
quotes_excel = pd.read_excel(
    excel_path,
    sheet_name="option_quotes",
    engine="openpyxl",
    dtype=dtype_map,
    parse_dates=date_columns,
)

print(quotes_excel.shape)
print(quotes_excel.dtypes)

(42, 18)
contract_id           string[python]
quote_date            datetime64[ns]
underlying            string[python]
currency              string[python]
option_type           string[python]
maturity_date         datetime64[ns]
time_to_maturity             float64
spot_price                   float64
strike_price                 float64
risk_free_rate               float64
dividend_yield               float64
implied_volatility           float64
bid_price                    float64
ask_price                    float64
mid_price                    float64
trading_volume                 int64
open_interest                  int64
data_source           string[python]
dtype: object


CSV와 Excel에서 읽은 호가가 같은지 검사합니다.

*강의록: `section03_data_io.tex` 809행 · 03-L039*

In [33]:
pd.testing.assert_frame_equal(
    quotes,
    quotes_excel,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("CSV and Excel contain the same table.")

CSV and Excel contain the same table.


<details><summary>강의록의 참고 출력 · 03-L040</summary>

```text
CSV and Excel contain the same table.
```

</details>

*강의록: `section03_data_io.tex` 832행 · 03-L041*

In [34]:
all_sheets = pd.read_excel(excel_path, sheet_name=None, engine="openpyxl")

print(all_sheets.keys())

dict_keys(['option_quotes', 'data_dictionary', 'README'])


## Excel 파일로 저장하기

DataFrame을 한 시트에 저장하는 기본 예제입니다. 다음 셀에서는 같은 실습용 출력 파일에 두 시트를 구성합니다.

*강의록: `section03_data_io.tex` 845행 · 03-L042*

In [35]:
output_excel_path = (output_dir / "option_data_analysis.xlsx")

call_quotes.to_excel(
    output_excel_path,
    sheet_name="call_quotes",
    index=False,
    engine="openpyxl",
)

**여러 시트를 가진 Excel 파일 만들기**

`ExcelWriter`를 사용하면 한 파일에 여러 시트를 저장할 수 있습니다.

*강의록: `section03_data_io.tex` 858행 · 03-L043*

In [36]:
summary_by_maturity = (
    quotes .groupby(["maturity_date", "option_type"], as_index=False)["mid_price"]
    .mean()
)

with pd.ExcelWriter(output_excel_path, engine="openpyxl") as writer:
    call_quotes.to_excel(writer, sheet_name="call_quotes", index=False)

    summary_by_maturity.to_excel(writer, sheet_name="summary", index=False)

In [37]:
print("저장한 시트:", pd.ExcelFile(output_excel_path).sheet_names)
summary_by_maturity

저장한 시트: ['call_quotes', 'summary']


,maturity_date,option_type,mid_price
0,2026-12-08,call,6.707237
1,2026-12-08,put,6.149589
2,2027-03-08,call,8.341649
3,2027-03-08,put,7.190230
4,2027-09-08,call,11.311091
5,2027-09-08,put,8.872753


## JSON과 Stata 파일

### JSON 파일

**DataFrame 일부를 JSON으로 저장하고 읽기**

작은 표를 JSON으로 저장하고 다시 읽습니다. API에 연결하지 않고도 입출력을 연습할 수 있습니다.

*강의록: `section03_data_io.tex` 910행 · 03-L044*

In [38]:
json_path = output_dir / "option_quotes_sample.json"

quotes.head(5).to_json(
    json_path,
    orient="records",
    date_format="iso",
    force_ascii=False,
    indent=2,
)

json_quotes = pd.read_json(json_path, orient="records")

json_quotes.head()

,contract_id,quote_date,underlying,currency,option_type,maturity_date,time_to_maturity,spot_price,strike_price,risk_free_rate,dividend_yield,implied_volatility,bid_price,ask_price,mid_price,trading_volume,open_interest,data_source
0,SYNTH100_20261208_C_080,2026-09-08T00:00:00.000,SYNTH100,KRW,call,2026-12-08T00:00:00.000,0.249315,100,80,0.035,0.0125,0.239763,20.409310,20.586972,20.498141,724,3924,synthetic_black_scholes
1,SYNTH100_20261208_P_080,2026-09-08T00:00:00.000,SYNTH100,KRW,put,2026-12-08T00:00:00.000,0.249315,100,80,0.035,0.0125,0.239763,0.101863,0.126646,0.114254,706,3804,synthetic_black_scholes
2,SYNTH100_20261208_C_090,2026-09-08T00:00:00.000,SYNTH100,KRW,call,2026-12-08T00:00:00.000,0.249315,100,90,0.035,0.0125,0.218311,11.245520,11.349466,11.297493,834,4054,synthetic_black_scholes
3,SYNTH100_20261208_P_090,2026-09-08T00:00:00.000,SYNTH100,KRW,put,2026-12-08T00:00:00.000,0.249315,100,90,0.035,0.0125,0.218311,0.814018,0.839433,0.826726,816,3934,synthetic_black_scholes
4,SYNTH100_20261208_C_095,2026-09-08T00:00:00.000,SYNTH100,KRW,call,2026-12-08T00:00:00.000,0.249315,100,95,0.035,0.0125,0.210787,7.368690,7.441279,7.404985,889,4119,synthetic_black_scholes


### Stata 파일

강의록의 `data/survey_data.dta`는 사용법을 설명하는 경로이고 원본 배포 자료에는 없습니다. 원래 구문은 아래에 보관하고, 별도 합성 자료를 `output/survey_data_demo.dta`에 만들어 같은 읽기 과정을 실행합니다.

*강의록: `section03_data_io.tex` 939행 · 03-L045*

```python
stata_data = pd.read_stata("data/survey_data.dta")
```

In [39]:
survey_demo = pd.DataFrame({
    "respondent_id": [1, 2, 3],
    "income": [3000.0, 4200.0, 3600.0],
    "age": [24, 31, 28],
})
stata_demo_path = output_dir / "survey_data_demo.dta"
survey_demo.to_stata(stata_demo_path, write_index=False)
stata_data = pd.read_stata(stata_demo_path)
print("교육용으로 만든 합성 자료입니다.")
stata_data

교육용으로 만든 합성 자료입니다.


,respondent_id,income,age
0,1,3000.0,24
1,2,4200.0,31
2,3,3600.0,28


방금 읽은 합성 Stata 표의 구조와 실제 자료형을 확인합니다.

*강의록: `section03_data_io.tex` 946행 · 03-L046*

In [40]:
stata_data.info()
print(stata_data.dtypes)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   respondent_id  3 non-null      int32  
 1   income         3 non-null      float64
 2   age            3 non-null      int32  
dtypes: float64(1), int32(2)
memory usage: 180.0 bytes
respondent_id      int32
income           float64
age                int32
dtype: object


## 직접 해보기

연습 1의 실수형 검사를 다시 수행합니다. 앞에서 만든 `float_columns`를 재사용합니다.

*강의록: `section03_data_io.tex` 1018행 · 03-L047*

In [41]:
assert (quotes[float_columns].dtypes == np.dtype(float)).all()

## 직접 해보기 · 풀이 예시

앞의 코드와 출력 결과를 이용해 먼저 직접 작성해 본 뒤 아래 풀이와 비교하세요.

### 연습 1 · CSV 구조와 자료형 확인

행·열 수, 앞뒤 다섯 행, 열 이름, 자료형, 결측·중복 개수를 확인합니다.

In [42]:
print("행·열:", quotes.shape)
display(quotes.head(), quotes.tail())
print("열 이름:", quotes.columns.tolist())
print(quotes.dtypes)
print("결측값:", int(quotes.isna().sum().sum()))
print("중복 계약:", int(quotes.duplicated(subset="contract_id").sum()))

행·열: (42, 18)


,contract_id,quote_date,underlying,currency,option_type,maturity_date,time_to_maturity,spot_price,strike_price,risk_free_rate,dividend_yield,implied_volatility,bid_price,ask_price,mid_price,trading_volume,open_interest,data_source
0,SYNTH100_20261208_C_080,2026-09-08,SYNTH100,KRW,call,2026-12-08,0.249315,100.0,80.0,0.035,0.0125,0.239763,20.409310,20.586972,20.498141,724,3924,synthetic_black_scholes
1,SYNTH100_20261208_P_080,2026-09-08,SYNTH100,KRW,put,2026-12-08,0.249315,100.0,80.0,0.035,0.0125,0.239763,0.101863,0.126646,0.114254,706,3804,synthetic_black_scholes
2,SYNTH100_20261208_C_090,2026-09-08,SYNTH100,KRW,call,2026-12-08,0.249315,100.0,90.0,0.035,0.0125,0.218311,11.245520,11.349466,11.297493,834,4054,synthetic_black_scholes
3,SYNTH100_20261208_P_090,2026-09-08,SYNTH100,KRW,put,2026-12-08,0.249315,100.0,90.0,0.035,0.0125,0.218311,0.814018,0.839433,0.826726,816,3934,synthetic_black_scholes
4,SYNTH100_20261208_C_095,2026-09-08,SYNTH100,KRW,call,2026-12-08,0.249315,100.0,95.0,0.035,0.0125,0.210787,7.368690,7.441279,7.404985,889,4119,synthetic_black_scholes


,contract_id,quote_date,underlying,currency,option_type,maturity_date,time_to_maturity,spot_price,strike_price,risk_free_rate,dividend_yield,implied_volatility,bid_price,ask_price,mid_price,trading_volume,open_interest,data_source
37,SYNTH100_20270908_P_105,2026-09-08,SYNTH100,KRW,put,2027-09-08,1.0,100.0,105.0,0.0375,0.0125,0.220716,9.976282,10.068401,10.022342,1134,4108,synthetic_black_scholes
38,SYNTH100_20270908_C_110,2026-09-08,SYNTH100,KRW,call,2027-09-08,1.0,100.0,110.0,0.0375,0.0125,0.217740,5.702461,5.764274,5.733367,1097,4293,synthetic_black_scholes
39,SYNTH100_20270908_P_110,2026-09-08,SYNTH100,KRW,put,2027-09-08,1.0,100.0,110.0,0.0375,0.0125,0.217740,12.869091,12.984856,12.926973,1079,4173,synthetic_black_scholes
40,SYNTH100_20270908_C_120,2026-09-08,SYNTH100,KRW,call,2027-09-08,1.0,100.0,120.0,0.0375,0.0125,0.215078,3.078736,3.124290,3.101513,987,4423,synthetic_black_scholes
41,SYNTH100_20270908_P_120,2026-09-08,SYNTH100,KRW,put,2027-09-08,1.0,100.0,120.0,0.0375,0.0125,0.215078,19.841190,20.012936,19.927063,969,4303,synthetic_black_scholes


열 이름: ['contract_id', 'quote_date', 'underlying', 'currency', 'option_type', 'maturity_date', 'time_to_maturity', 'spot_price', 'strike_price', 'risk_free_rate', 'dividend_yield', 'implied_volatility', 'bid_price', 'ask_price', 'mid_price', 'trading_volume', 'open_interest', 'data_source']
contract_id           string[python]
quote_date            datetime64[ns]
underlying            string[python]
currency              string[python]
option_type           string[python]
maturity_date         datetime64[ns]
time_to_maturity             float64
spot_price                   float64
strike_price                 float64
risk_free_rate               float64
dividend_yield               float64
implied_volatility           float64
bid_price                    float64
ask_price                    float64
mid_price                    float64
trading_volume                 int64
open_interest                  int64
data_source           string[python]
dtype: object
결측값: 0
중복 계약: 0


### 연습 2 · 1년 만기 콜옵션 저장

만기가 `1.0`인 콜을 선택하고 특징 변수를 추가한 뒤, 저장 전후의 표를 비교합니다.

In [43]:
one_year_calls = quotes.loc[
    (quotes["option_type"] == "call") & np.isclose(quotes["time_to_maturity"], 1.0)
].copy()
one_year_calls["bid_ask_spread"] = one_year_calls["ask_price"] - one_year_calls["bid_price"]
one_year_calls["moneyness"] = one_year_calls["strike_price"] / one_year_calls["spot_price"]
one_year_path = output_dir / "one_year_call_quotes.csv"
one_year_calls.to_csv(one_year_path, index=False, encoding="utf-8-sig")
reloaded_one_year = pd.read_csv(
    one_year_path, encoding="utf-8-sig",
    dtype={**dtype_map, "bid_ask_spread": float, "moneyness": float},
    parse_dates=date_columns,
)
pd.testing.assert_frame_equal(
    one_year_calls.reset_index(drop=True), reloaded_one_year,
    check_exact=False, rtol=1e-12, atol=1e-12,
)
print("1년 만기 콜 행 수:", len(reloaded_one_year))
print(reloaded_one_year.dtypes)
reloaded_one_year.head()

1년 만기 콜 행 수: 7
contract_id           string[python]
quote_date            datetime64[ns]
underlying            string[python]
currency              string[python]
option_type           string[python]
maturity_date         datetime64[ns]
time_to_maturity             float64
spot_price                   float64
strike_price                 float64
risk_free_rate               float64
dividend_yield               float64
implied_volatility           float64
bid_price                    float64
ask_price                    float64
mid_price                    float64
trading_volume                 int64
open_interest                  int64
data_source           string[python]
bid_ask_spread               float64
moneyness                    float64
dtype: object


,contract_id,quote_date,underlying,currency,option_type,maturity_date,time_to_maturity,spot_price,strike_price,risk_free_rate,dividend_yield,implied_volatility,bid_price,ask_price,mid_price,trading_volume,open_interest,data_source,bid_ask_spread,moneyness
0,SYNTH100_20270908_C_080,2026-09-08,SYNTH100,KRW,call,2027-09-08,1.0,100.0,80.0,0.0375,0.0125,0.259763,23.644740,23.846759,23.745749,987,3903,synthetic_black_scholes,0.202019,0.80
1,SYNTH100_20270908_C_090,2026-09-08,SYNTH100,KRW,call,2027-09-08,1.0,100.0,90.0,0.0375,0.0125,0.238311,16.035310,16.175315,16.105313,1097,4033,synthetic_black_scholes,0.140004,0.90
2,SYNTH100_20270908_C_095,2026-09-08,SYNTH100,KRW,call,2027-09-08,1.0,100.0,95.0,0.0375,0.0125,0.230787,12.780680,12.894012,12.837346,1152,4098,synthetic_black_scholes,0.113332,0.95
3,SYNTH100_20270908_C_100,2026-09-08,SYNTH100,KRW,call,2027-09-08,1.0,100.0,100.0,0.0375,0.0125,0.225000,9.964607,10.054679,10.009643,1207,4163,synthetic_black_scholes,0.090072,1.00
4,SYNTH100_20270908_C_105,2026-09-08,SYNTH100,KRW,call,2027-09-08,1.0,100.0,105.0,0.0375,0.0125,0.220716,7.607565,7.681851,7.644708,1152,4228,synthetic_black_scholes,0.074287,1.05


### 연습 3 · 잘못된 저장 옵션 진단

앞서 만든 두 파일을 다시 읽어 불필요한 인덱스 열과 금리의 변화를 확인합니다.

In [44]:
index_check = pd.read_csv(bad_index_path, encoding="utf-8-sig")
precision_check = pd.read_csv(bad_rate_path)
print("불필요한 첫 열:", index_check.columns[0])
print("원래 금리:", rate_example.loc[0, "risk_free_rate"])
print("저장 후 금리:", precision_check.loc[0, "risk_free_rate"])

불필요한 첫 열: Unnamed: 0
원래 금리: 0.035125
저장 후 금리: 0.04


### 연습 4 · 풋옵션 표와 요약표를 Excel에 저장

원본 Excel은 보존하고 `output/put_option_analysis.xlsx`에 두 시트를 만듭니다.

In [45]:
put_quotes = quotes.loc[quotes["option_type"] == "put"].copy()
put_excel_path = output_dir / "put_option_analysis.xlsx"
with pd.ExcelWriter(put_excel_path, engine="openpyxl") as writer:
    put_quotes.to_excel(writer, sheet_name="put_quotes", index=False)
    summary_by_maturity.to_excel(writer, sheet_name="summary", index=False)
print("저장한 시트:", pd.ExcelFile(put_excel_path).sheet_names)
print("풋옵션 행 수:", len(pd.read_excel(put_excel_path, sheet_name="put_quotes")))

저장한 시트: ['put_quotes', 'summary']
풋옵션 행 수: 21


### 연습 5 · ATM 옵션을 JSON으로 저장

행사가격이 `100.0`인 옵션을 선택하고 읽은 뒤 행 수를 확인합니다.

In [46]:
atm_quotes = quotes.loc[np.isclose(quotes["strike_price"], 100.0)].copy()
atm_json_path = output_dir / "at_the_money_quotes.json"
atm_quotes.to_json(
    atm_json_path, orient="records", date_format="iso",
    force_ascii=False, indent=2,
)
reloaded_atm = pd.read_json(atm_json_path, orient="records")
print("ATM 옵션 행 수:", len(reloaded_atm))
reloaded_atm[["contract_id", "option_type", "strike_price", "mid_price"]]

ATM 옵션 행 수: 6


,contract_id,option_type,strike_price,mid_price
0,SYNTH100_20261208_C_100,call,100,4.342600
1,SYNTH100_20261208_P_100,put,100,3.784952
2,SYNTH100_20270308_C_100,call,100,6.555598
3,SYNTH100_20270308_P_100,put,100,5.404178
4,SYNTH100_20270908_C_100,call,100,10.009643
5,SYNTH100_20270908_P_100,put,100,7.571305


## 확인할 내용

- CSV·Excel 파일이 있다는 것과 읽은 표가 올바르다는 것은 다릅니다. 자료형과 값의 조건을 함께 검사합니다.
- 원본은 `data`, 새 결과는 `output`에 보존합니다.
- `index=False`와 충분한 저장 정밀도를 유지하고 저장 후 재검증합니다.
- 여러 시트가 필요한 경우에는 `ExcelWriter`를 사용합니다.

다음 파일: [Section 4 · 그래프](section04_plotting.ipynb)